In [10]:
import json
import shutil
import hashlib
from pathlib import Path


def convert_coco_to_yolo(
    coco_json_path,
    images_dir,
    dataset_root,
    split="train",
    copy_images=True,
    max_name_length=100,
):
    """
    Convert COCO dataset to Ultralytics YOLO format (robust + Windows-safe).

    Parameters
    ----------
    coco_json_path : str
        Path to COCO JSON file.

    images_dir : str
        Directory containing original images.

    dataset_root : str
        Root directory for YOLO dataset.

    split : str
        Dataset split name (train/val/test).

    copy_images : bool
        If True, copy images into YOLO images/<split>/.

    max_name_length : int
        Maximum allowed filename length before hashing.
    """

    coco_json_path = Path(coco_json_path)
    images_dir = Path(images_dir)
    dataset_root = Path(dataset_root)

    images_out = dataset_root / "images" / split
    labels_out = dataset_root / "labels" / split

    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)

    with open(coco_json_path) as f:
        coco = json.load(f)

    images = {img["id"]: img for img in coco["images"]}
    annotations = coco["annotations"]

    anns_per_image = {}
    for ann in annotations:
        anns_per_image.setdefault(ann["image_id"], []).append(ann)

    converted = 0
    skipped_missing_images = 0
    skipped_invalid_boxes = 0

    for image_id, img_info in images.items():
        file_name = img_info["file_name"]
        image_path = images_dir / file_name

        # Skip if image file does not exist
        if not image_path.exists():
            skipped_missing_images += 1
            continue

        img_w = img_info["width"]
        img_h = img_info["height"]

        # ---------------------------
        # SAFE FILENAME HANDLING
        # ---------------------------

        base_name = Path(file_name).stem

        # Remove Roboflow hash if present
        if ".rf." in base_name:
            base_name = base_name.split(".rf.")[0]

        # If still too long, hash it
        if len(base_name) > max_name_length:
            base_name = hashlib.md5(base_name.encode()).hexdigest()

        label_path = labels_out / f"{base_name}.txt"
        image_out_path = images_out / f"{base_name}{Path(file_name).suffix}"

        # Ensure parent directory exists
        label_path.parent.mkdir(parents=True, exist_ok=True)

        with open(label_path, "w") as f:
            for ann in anns_per_image.get(image_id, []):
                x, y, w, h = ann["bbox"]

                # Skip invalid boxes
                if w <= 0 or h <= 0:
                    skipped_invalid_boxes += 1
                    continue

                # Convert to YOLO normalized format
                x_center = (x + w / 2) / img_w
                y_center = (y + h / 2) / img_h
                w /= img_w
                h /= img_h

                # COCO category_id is usually 1-indexed
                class_id = ann["category_id"] - 1

                f.write(f"{class_id} {x_center} {y_center} {w} {h}\n")

        # Optionally copy image
        if copy_images:
            shutil.copy2(image_path, image_out_path)

        converted += 1

    print("Conversion complete.")
    print(f"Images converted: {converted}")
    print(f"Missing images skipped: {skipped_missing_images}")
    print(f"Invalid boxes skipped: {skipped_invalid_boxes}")


In [11]:
coco_json_path = r"C:\Users\chris\Desktop\University\Thesis\ExternalDatasets\MicrogliaIdentificationDataset\train-20230405T084905Z-001\train\train_annotations.json"
images_dir = r"C:\Users\chris\Desktop\University\Thesis\ExternalDatasets\MicrogliaIdentificationDataset\train-20230405T084905Z-001\train"
output_dir = r"C:\Users\chris\Desktop\University\Thesis\AutomaticMicrogliaMorphologyAnalysis\initial_pipeline\object_detection\custom_detection\rat_yolo_dataset"

convert_coco_to_yolo(coco_json_path=coco_json_path, images_dir=images_dir, dataset_root=output_dir)

Conversion complete.
Images converted: 602
Missing images skipped: 0
Invalid boxes skipped: 0
